In [2]:
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import mesh, fem, default_scalar_type

domain = mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0.0, 0.0]), np.array([2.0, 1.0])],   # lower-left, upper-right
    [32, 16],                                        # cells in x, y
    cell_type=mesh.CellType.triangle,
)

V = fem.functionspace(domain, ("Lagrange", 1))

# Step 1: Write marker functions
These are functions that receive a point, and return a boolean to determine if the point is in the domain of the mesh entity.

In [3]:
def left(x):
    return np.isclose(x[0], 0.0)

def right(x):
    return np.isclose(x[0], 2.0)

def bottom(x):
    return np.isclose(x[1], 0.0)

def top(x):
    return np.isclose(x[1], 1.0)

Compund regions can be found using boolean algebra, using the python functions `np.logical_and()` or `np.logical_or()`

# Step 2: Find the facets on each boundary 
DOFs live on mesh entitites, so first we need to find the _facets_ (in 2D these are edges, in 3D they would be surfaces) that lie on each side.

In [4]:
tdim = domain.topology.dim        # 2
fdim = tdim - 1                   # 1
domain.topology.create_connectivity(fdim, tdim)

left_facets   = mesh.locate_entities_boundary(domain, fdim, left)
right_facets  = mesh.locate_entities_boundary(domain, fdim, right)
bottom_facets = mesh.locate_entities_boundary(domain, fdim, bottom)
top_facets    = mesh.locate_entities_boundary(domain, fdim, top)

# Step 3, Turn the facets into DOFs
This step involves asking V's degree-of-freedom map: "which degress of fredom (nodes and their dofs) are attached to these facets?" 


In [5]:
left_dofs   = fem.locate_dofs_topological(V, fdim, left_facets)
right_dofs  = fem.locate_dofs_topological(V, fdim, right_facets)
bottom_dofs = fem.locate_dofs_topological(V, fdim, bottom_facets)
top_dofs    = fem.locate_dofs_topological(V, fdim, top_facets)

In [7]:
left_dofs, top_dofs

(array([424, 425, 441, 456, 470, 483, 495, 506, 516, 525, 533, 540, 546,
        551, 555, 558, 560], dtype=int32),
 array([152, 168, 185, 202, 219, 236, 253, 270, 287, 304, 321, 338, 355,
        372, 389, 406, 423, 440, 455, 469, 482, 494, 505, 515, 524, 532,
        539, 545, 550, 554, 557, 559, 560], dtype=int32))

# Using tags instead.
For boundary integrals, it might be better to tag the boundaries and reuse them for different Dirichlet and Neumann BCs.

In [8]:
boundaries = [(1, left), (2, right), (3, bottom), (4, top)]

indices, markers = [], []
for tag, locator in boundaries:
    facets = mesh.locate_entities_boundary(domain, fdim, locator)
    indices.append(facets)
    markers.append(np.full_like(facets, tag))

In [16]:
indices = np.hstack(indices).astype(np.int32)
markers = np.hstack(markers).astype(np.int32)
order = np.argsort(indices)
ft = mesh.meshtags(domain, fdim, indices[order], markers[order])

In [21]:
ds = ufl.Measure("ds", domain=domain, subdomain_data=ft)
ds

Measure('ds', subdomain_id='everywhere', domain=Mesh(blocked element (Basix element (P, triangle, 1, gll_warped, unset, False, float64, []), (2,)), 0), subdomain_data=<dolfinx.mesh.MeshTags object at 0xffff4b0c4c20>)

In [22]:
import gmsh